In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F

#### Passo 2.1 - Renomear os sete campos essenciais

In [0]:
train = spark.table(TABLE_BRONZE_TRAIN)
test = spark.table(TABLE_BRONZE_TEST)

train_named = (
    train
    .withColumnRenamed('trans_num', 'tx_id')
    .withColumnRenamed('cc_num', 'card_id')
    .withColumnRenamed('trans_date_trans_time', 'event_ts_text')
    .withColumnRenamed('amt', 'amount_text')
    .withColumnRenamed('is_fraud', 'label_text')
)

test_named = (
    test
    .withColumnRenamed('trans_num', 'tx_id')
    .withColumnRenamed('cc_num', 'card_id')
    .withColumnRenamed('trans_date_trans_time', 'event_ts_text')
    .withColumnRenamed('amt', 'amount_text')
    .withColumnRenamed('is_fraud', 'label_text')
)

#### Passo 2.2 - Converter tipos e tornar falhas visíveis

In [0]:
train_typed = (
    train_named
    .withColumn(
        'event_ts',
        F.to_timestamp('event_ts_text','yyyy-MM-dd HH:mm:ss')
    )
    .withColumn(
        'amount',
        F.expr('try_cast(amount_text as double)')
        )
    .withColumn(
        'is_fraud',
        F.expr('try_cast(label_text as int)')
    )

)

test_typed = (
    test_named
    .withColumn(
        'event_ts',
        F.to_timestamp('event_ts_text', 'yyyy-MM-dd HH:mm:ss')
    )
    .withColumn(
        'amount',
        F.expr('try_cast(amount_text as double)')
    )
    .withColumn(
        'is_fraud',
        F.expr('try_cast(label_text as int)')
    )
)

#### Passo 2.3 - Contar chaves ausentes, timestamp inválido e labels inválidos

No train, conte: tx_id ausente/vazio, tx_id duplicado, event_ts inválido, amount inválido/negativo e is_fraud fora de 0/1. Faça primeiro no train. Depois repita no test usando o mesmo raciocínio.

In [0]:
total_train = train_typed.count()

tx_distintos_train = (
    train_typed
    .select("tx_id")
    .distinct()
    .count()
)

print(
    "Train - tx_id duplicados:",
    total_train - tx_distintos_train
)


total_test = test_typed.count()

tx_distintos_test = (
    test_typed
    .select("tx_id")
    .distinct()
    .count()
)

print(
    "Test - tx_id duplicados:",
    total_test - tx_distintos_test
)

In [0]:
print(
    "Train - tx_id ausente/vazio:",
    (
        train_typed
        .filter(
            F.col("tx_id").isNull()
            | (F.trim(F.col("tx_id")) == "")
        )
        .count()
    )
)

print(
    "Test - tx_id ausente/vazio:",
    (
        test_typed
        .filter(
            F.col("tx_id").isNull()
            | (F.trim(F.col("tx_id")) == "")
        )
        .count()
    )
)

In [0]:
print(
    "Train - event_ts inválido:",
    (
        train_typed
        .filter(
            F.col("event_ts_text").isNotNull()
            & F.col("event_ts").isNull()
        )
        .count()
    )
)

print(
    "Test - event_ts inválido:",
    (
        test_typed
        .filter(
            F.col("event_ts_text").isNotNull()
            & F.col("event_ts").isNull()
        )
        .count()
    )
)

In [0]:
print(
    "Train - amount inválido ou negativo:",
    (
        train_typed
        .filter(
            (
                F.col("amount_text").isNotNull()
                & F.col("amount").isNull()
            )
            | (F.col("amount") < 0)
        )
        .count()
    )
)

print(
    "Test - amount inválido ou negativo:",
    (
        test_typed
        .filter(
            (
                F.col("amount_text").isNotNull()
                & F.col("amount").isNull()
            )
            | (F.col("amount") < 0)
        )
        .count()
    )
)

In [0]:
print(
    "Train - is_fraud inválido:",
    (
        train_typed
        .filter(
            F.col("is_fraud").isNull()
            | (~F.col("is_fraud").isin(0, 1))
        )
        .count()
    )
)

print(
    "Test - is_fraud inválido:",
    (
        test_typed
        .filter(
            F.col("is_fraud").isNull()
            | (~F.col("is_fraud").isin(0, 1))
        )
        .count()
    )
)

#### Passo 2.4 - Escolher as colunas da Silver

Crie train_silver e test_silver com tx_id, card_id, event_ts, merchant, category, amount, is_fraud e source. Mantenha também as três colunas *_text por enquanto para rastreabilidade.

In [0]:
train_typed

In [0]:
silver_cols = [
    "tx_id", "card_id", "event_ts", "merchant", "category",
    "amount", "is_fraud",
    "event_ts_text", "amount_text", "label_text",
]

train_silver = train_typed.select(*silver_cols)
test_silver = test_typed.select(*silver_cols)

#### Passo 2.5 - Gravar Silver somente após os controles

In [0]:
(
    train_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABLE_SILVER_TRAIN)
)

(
    test_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABLE_SILVER_TEST)
)

spark.table(TABLE_SILVER_TRAIN).printSchema()